In [1]:
def main(datasources, start_date, end_date):
    """AI_LSR_01：可见深度状态路由的价格路径因子。

    每日收盘后计算三个动态中证1000截面秩：日内振幅R、早晚价格路径P、
    五档可见深度D。若D不低于当日中位数，factor=-R；否则factor=-P。
    一分钟五档仅解释为可见盘口状态，不代表真实订单流、撤单或主动方向。
    """
    import gc
    import numpy as np
    import pandas as pd
    import dai

    bar1m_table = datasources["bar1m"]
    start = pd.Timestamp(start_date).normalize()
    end = pd.Timestamp(end_date).normalize()
    if end < start:
        raise ValueError("end_date 不能早于 start_date")

    def normalize_keys(frame):
        result = frame.copy()
        result["date"] = pd.to_datetime(
            result["date"], errors="coerce"
        ).dt.normalize()
        result["instrument"] = result["instrument"].astype(str).str.strip()
        if result["date"].isna().any():
            raise ValueError("date 存在无法解析的值")
        if result["instrument"].eq("").any():
            raise ValueError("instrument 存在空字符串")
        return result

    def month_windows():
        cursor = start.replace(day=1)
        windows = []
        while cursor <= end:
            next_month = cursor + pd.offsets.MonthBegin(1)
            windows.append(
                (max(start, cursor), min(end, next_month - pd.Timedelta(days=1)))
            )
            cursor = next_month
        return windows

    def query_month(month_start, month_end):
        # 服务端先压缩到股票日，只拉取冻结表达式需要的三个底座特征。
        sql = f"""
        WITH minute_base AS (
            SELECT
                date,
                date::DATE::DATETIME AS trading_day,
                instrument,
                STRFTIME(date, '%H:%M') AS hm,
                CAST(high AS DOUBLE) AS px_high,
                CAST(low AS DOUBLE) AS px_low,
                CAST(close AS DOUBLE) AS px_close,
                CASE
                    WHEN ask_price1 > 0 AND bid_price1 > 0
                    THEN (
                        CAST(COALESCE(bid_volume1, 0) AS DOUBLE)
                      + CAST(COALESCE(bid_volume2, 0) AS DOUBLE)
                      + CAST(COALESCE(bid_volume3, 0) AS DOUBLE)
                      + CAST(COALESCE(bid_volume4, 0) AS DOUBLE)
                      + CAST(COALESCE(bid_volume5, 0) AS DOUBLE)
                      + CAST(COALESCE(ask_volume1, 0) AS DOUBLE)
                      + CAST(COALESCE(ask_volume2, 0) AS DOUBLE)
                      + CAST(COALESCE(ask_volume3, 0) AS DOUBLE)
                      + CAST(COALESCE(ask_volume4, 0) AS DOUBLE)
                      + CAST(COALESCE(ask_volume5, 0) AS DOUBLE)
                    )
                    ELSE NULL
                END AS visible_depth
            FROM {bar1m_table}
        )
        SELECT
            trading_day AS date,
            instrument,
            LAST(px_close ORDER BY date) AS day_close,
            MAX(px_high) AS day_high,
            MIN(px_low) AS day_low,
            AVG(CASE WHEN hm BETWEEN '09:31' AND '10:00' THEN px_close END)
                AS early_price,
            AVG(CASE WHEN hm BETWEEN '14:31' AND '15:00' THEN px_close END)
                AS late_price,
            AVG(visible_depth) AS depth_mean
        FROM minute_base
        GROUP BY trading_day, instrument
        """
        filters = {
            "date": [
                month_start.strftime("%Y-%m-%d 00:00:00"),
                month_end.strftime("%Y-%m-%d 23:59:59"),
            ]
        }
        daily = dai.query(sql, filters=filters, compression=True).df()
        pool = dai.query(
            """
            SELECT date, instrument
            FROM bigalpha_2026_instruments
            """,
            filters=filters,
            compression=True,
        ).df()
        daily = normalize_keys(daily)
        pool = normalize_keys(pool)
        if daily.duplicated(["date", "instrument"]).any():
            raise ValueError("分钟聚合结果存在重复键")
        if pool.duplicated(["date", "instrument"]).any():
            raise ValueError("动态股票池存在重复键")
        return pool.merge(
            daily, how="left", on=["date", "instrument"], validate="one_to_one"
        )

    pieces = []
    for month_start, month_end in month_windows():
        pieces.append(query_month(month_start, month_end))
        gc.collect()
    base = pd.concat(pieces, ignore_index=True)
    del pieces

    numeric = [
        "day_close", "day_high", "day_low", "early_price", "late_price",
        "depth_mean",
    ]
    for column in numeric:
        base[column] = pd.to_numeric(base[column], errors="coerce")
    base[numeric] = base[numeric].replace([np.inf, -np.inf], np.nan)

    base["range_relative"] = (
        (base["day_high"] - base["day_low"])
        / base["day_close"].where(base["day_close"].abs().gt(1e-12))
    )
    base["early_late_return"] = (
        base["late_price"]
        / base["early_price"].where(base["early_price"].abs().gt(1e-12))
        - 1.0
    )
    base["log_visible_depth"] = np.log1p(base["depth_mean"].clip(lower=0))
    features = ["range_relative", "early_late_return", "log_visible_depth"]
    base[features] = base[features].replace([np.inf, -np.inf], np.nan)

    # 先执行冻结时的底座硬门，再以当日中位数表示未知的中性状态。
    grouped = base.groupby("date", observed=True, sort=False)
    stock_count = grouped["instrument"].size()
    for feature in features:
        valid_count = grouped[feature].count()
        missing_rate = 1.0 - valid_count / stock_count
        unique_count = grouped[feature].nunique(dropna=True)
        daily_std = grouped[feature].std(ddof=1).fillna(0.0)
        if (
            valid_count.min() < 800
            or missing_rate.max() > 0.20
            or unique_count.min() < 200
            or daily_std.min() <= 0.0
        ):
            raise ValueError(f"底座特征未通过冻结覆盖硬门：{feature}")
        median = grouped[feature].transform("median")
        neutral = base[feature].fillna(median)
        rank = neutral.groupby(base["date"], observed=True).rank(
            method="average", pct=True
        )
        base[f"{feature}_rank"] = (rank - 0.5) * 2.0

    # 冻结基因：template=4, a=range, b=early_late, c=depth；方向=-1。
    base["factor"] = -np.where(
        base["log_visible_depth_rank"].ge(0.0),
        base["range_relative_rank"],
        base["early_late_return_rank"],
    )
    result = base[["date", "instrument", "factor"]].copy()
    result["factor"] = pd.to_numeric(result["factor"], errors="coerce").replace(
        [np.inf, -np.inf], np.nan
    )
    result = result.sort_values(["date", "instrument"]).reset_index(drop=True)

    if result.duplicated(["date", "instrument"]).any():
        raise ValueError("输出存在重复的 date/instrument 键")
    quality = result.groupby("date", observed=True)["factor"].agg(
        valid_count="count", unique_count="nunique", daily_std="std"
    )
    daily_size = result.groupby("date", observed=True).size()
    if (1.0 - quality["valid_count"] / daily_size).max() > 0.40:
        raise ValueError("输出至少一个交易日缺失率超过40%")
    if quality["valid_count"].min() < 950:
        raise ValueError("输出至少一个交易日有效股票少于950")
    if quality["unique_count"].min() < 200 or quality["daily_std"].min() <= 0:
        raise ValueError("输出至少一个交易日截面变化不足")
    return result
